In [1]:
%pip install -U torch torchvision transformers datasets evaluate accelerate scikit-learn emoji peft bitsandbytes

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

import re

In [3]:
temp = load_dataset("csv", data_files="/content/drive/MyDrive/Sentiment140.csv", encoding="ISO-8859-1")["train"]
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]] }

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=1)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=1)

#EXPERIMENTAL SAMPLE
dataset_dict = {
    "train": temp["train"].shuffle(seed=1).select(range(2000)), # EXPERIMENTAL SAMPLE
    "test": test_valid["test"].select(range(500)),
    "validation": test_valid["train"].select(range(500)),
}

dataset_dict

{'train': Dataset({
     features: ['Label', 'Text'],
     num_rows: 2000
 }),
 'test': Dataset({
     features: ['Label', 'Text'],
     num_rows: 500
 }),
 'validation': Dataset({
     features: ['Label', 'Text'],
     num_rows: 500
 })}

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
model_path = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

In [9]:
def preprocess_function(x):
    return tokenizer(x["Text"], truncation=True, padding=False, max_length=128)

tokenized = {}
for split, ds in dataset_dict.items():
    ds = ds.rename_column("Label", "labels")
    tokenized[split] = ds.map(preprocess_function, batched=True)

In [7]:
for n, m in model.named_modules():
    if any(k in n.lower() for k in ("q_proj","v_proj","query","key","value","dense","output","proj","attn")):
        print(n)

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

roberta.encoder.layer.0.attention.self.query
roberta.encoder.layer.0.attention.self.key
roberta.encoder.layer.0.attention.self.value
roberta.encoder.layer.0.attention.output
roberta.encoder.layer.0.attention.output.dense
roberta.encoder.layer.0.attention.output.LayerNorm
roberta.encoder.layer.0.attention.output.dropout
roberta.encoder.layer.0.intermediate.dense
roberta.encoder.layer.0.output
roberta.encoder.layer.0.output.dense
roberta.encoder.layer.0.output.LayerNorm
roberta.encoder.layer.0.output.dropout
roberta.encoder.layer.1.attention.self.query
roberta.encoder.layer.1.attention.self.key
roberta.encoder.layer.1.attention.self.value
roberta.encoder.layer.1.attention.output
roberta.encoder.layer.1.attention.output.dense
roberta.encoder.layer.1.attention.output.LayerNorm
roberta.encoder.layer.1.attention.output.dropout
roberta.encoder.layer.1.intermediate.dense
roberta.encoder.layer.1.output
roberta.encoder.layer.1.output.dense
roberta.encoder.layer.1.output.LayerNorm
roberta.encoder

In [8]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 887,042 || all params: 135,788,548 || trainable%: 0.6533


In [9]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

True Tesla T4


In [10]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback
import evaluate, numpy as np

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir="outputs/lora-bertweet",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",   # important: disables integration callbacks that often trigger this
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Remove notebook progress callback (common source of this exact error on Colab)
trainer.remove_callback(NotebookProgressCallback)

trainer.train()
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results["eval_accuracy"])

Test accuracy: 0.804


In [11]:
model.save_pretrained('/content/drive/MyDrive/lora-bertweet-sentiment')
tokenizer.save_pretrained('/content/drive/MyDrive/lora-bertweet-sentiment')

('/content/drive/MyDrive/lora-bertweet-sentiment/tokenizer_config.json',
 '/content/drive/MyDrive/lora-bertweet-sentiment/vocab.txt',
 '/content/drive/MyDrive/lora-bertweet-sentiment/bpe.codes',
 '/content/drive/MyDrive/lora-bertweet-sentiment/added_tokens.json')

In [10]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from transformers.utils.notebook import NotebookProgressCallback
from peft import PeftConfig, PeftModel
import evaluate
import numpy as np

model_dir = "/content/drive/MyDrive/lora-bertweet-sentiment"

# Read adapter config to find correct base model (e.g., vinai/bertweet-base)
peft_config = PeftConfig.from_pretrained(model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)  # or peft_config.base_model_name_or_path

base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

model = PeftModel.from_pretrained(base_model, model_dir)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

eval_args = TrainingArguments(
    output_dir="outputs/eval-lora-bertweet",
    per_device_eval_batch_size=64,
    report_to="none",   # avoid callback integration issues
    do_train=False,
    do_eval=True,
)

trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Avoid notebook callback state bug in Colab
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.decoder.bias            | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.pooler.dense.weight     | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
roberta.pooler.dense.bias       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initi

Test accuracy: 0.626
Test F1: 0.7391910739191074
